# Bodhan Tool Server and Task Agent — Guided Walkthrough

Welcome to the guided walkthrough for the **Bodhan MCP Tool Server (`bodhan_mcp`) and Autonomous Task Agent (`bodhan_agent`)**.

This notebook demonstrates:
1. **Standalone MCP Server (`bodhan_mcp`)**: 11 typed tools, 3 resources (`bodhan://voices`, `bodhan://languages`, `bodhan://pricing`), 2 prompts (`prompts/list` + `prompts/get`), and strict `inputSchema` validation.
2. **Media Plumbing & Language Detection**: Inspecting files/directories with upfront INR cost estimates and bounded untrusted text reading (`bodhan_read_text`).
3. **All 4 Bodhan Models with Internal Limit Handling**:
   - `bodhan_transcribe` (25s WAV chunking + `ffmpeg` container conversion + SRT stitching)
   - `bodhan_translate` (table-preserving chunking, split-and-retry on truncation, canonical script variants like `ks-Deva`, refuses ASR-only `bhb`/`bho`)
   - `bodhan_speak` (220-char sentence/clause splitting, PCM16 WAV stitching, voice/language validation, cache-bypassing `verify=True` repair retry)
   - `bodhan_ocr` (images, folders, and multi-page PDFs via pinned `pypdfium2`, furniture block filtering)
4. **First-Class Verification & Subtitles**: `bodhan_verify_translation`, `bodhan_verify_speech`, and `bodhan_make_subtitles` (42 chars × 2 lines SRT/VTT).
5. **Autonomous Task Agent, Security Guards & Budgets**: Ground Rule 1 Indic provenance enforcement in code, path traversal blocking, upfront `--max-cost-inr` protection, and honest `impossible_task` handling.

## 1. Setup & Environment

Set `BODHAN_MOCK=1` to run completely offline with deterministic mock responses, or leave `BODHAN_MOCK=0` with your Bodhan & LLM keys in `.env` to run live against `api.bodhan.ai`.

In [ ]:
import json
import os
import sys
from pathlib import Path

# Ensure recipe src/ is on sys.path whether opened from repo root or recipe folder
RECIPE_DIR = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd() / "examples" / "bodhan_tool_agent"
SRC = (RECIPE_DIR / "src").resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Uncomment below to force offline mock mode for zero-cost walkthrough runs:
# os.environ["BODHAN_MOCK"] = "1"

from bodhan_mcp.client import BodhanClient

client = BodhanClient()
print(f"Recipe dir: {RECIPE_DIR}")
print(f"Bodhan Client initialized | mock_mode={client.mock_mode} | disk_cache={client.cache.cache_dir}")

## 2. Standalone MCP Server Protocol (`bodhan_mcp`)

Any off-the-shelf MCP client (Claude Desktop, Cursor, MCP Inspector) can connect over stdio (`python -m bodhan_mcp`) or token-authenticated SSE (`python -m bodhan_mcp --transport sse --port 8080 --token ...`). Below we exercise `initialize`, `tools/list`, `resources/read`, `prompts/get`, and structured `INVALID_ARGUMENTS` error handling directly.

In [ ]:
from bodhan_mcp.server import MCPServer

server = MCPServer()

init_resp = server.handle_request({"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {}})
tools_resp = server.handle_request({"jsonrpc": "2.0", "id": 2, "method": "tools/list", "params": {}})
prompt_resp = server.handle_request({
    "jsonrpc": "2.0", "id": 3, "method": "prompts/get",
    "params": {"name": "transcribe_and_subtitle", "arguments": {"audio_file": "sample_hindi.wav"}},
})
bad_call = server.handle_request({
    "jsonrpc": "2.0", "id": 4, "method": "tools/call",
    "params": {"name": "bodhan_translate", "arguments": {"text": "Hello"}},
})

print("Server:", init_resp["result"]["serverInfo"])
print("Registered tools (11):", [t["name"] for t in tools_resp["result"]["tools"]])
print("Prompt template:", prompt_resp["result"]["messages"][0]["content"]["text"])
print("Bad-input error:", bad_call["result"]["content"][0]["text"])

## 3. Media Inspection, Language Detection & Untrusted Text Reading

Inspect files or folders with upfront INR cost estimates without loading binaries into LLM context, detect Indian vs. unsupported scripts, and read bounded text slices with prompt-injection warnings.

In [ ]:
from bodhan_mcp.tools import bodhan_inspect_media, bodhan_detect_language, bodhan_read_text, bodhan_list_voices

sample_wav = str(RECIPE_DIR / "sample_data" / "sample_hindi.wav")
sample_pdf = str(RECIPE_DIR / "sample_data" / "chapter_oco3.pdf")
sample_txt = str(RECIPE_DIR / "sample_data" / "bengali_para.txt")

for path in (sample_wav, sample_pdf, sample_txt):
    info = bodhan_inspect_media(path)
    print(f"{Path(info.path).name}: type={info.media_type} | {info.notes}")

print("\nTamil detection:", bodhan_detect_language(text="வணக்கம்! இது தமிழ் உரை.").model_dump())
print("Arabic detection (unsupported):", bodhan_detect_language(text="مرحبا بالعالم هذه لغة عربية").model_dump())
print("Bengali voices:", [(v.name, v.gender) for v in bodhan_list_voices("bn")])

slice_res = bodhan_read_text(sample_txt, offset_chars=0, max_chars=120)
print(f"\nbodhan_read_text -> untrusted={slice_res['untrusted']}, returned={slice_res['returned_chars']}/{slice_res['total_chars']} chars, warnings={slice_res['injection_warnings']}")

## 4. Speech-to-Text (`bodhan_transcribe`) & Subtitle Formatting (`bodhan_make_subtitles`)

`bodhan_transcribe` chunks audio > 25s into WAV windows and converts non-WAV containers (`.mp3`, `.mp4`, `.ogg`, `.m4a`, `.webm`) via `ffmpeg`. `bodhan_make_subtitles` wraps cues cleanly to 42 chars × 2 lines in SRT or VTT.

In [ ]:
from bodhan_mcp.tools import bodhan_transcribe, bodhan_make_subtitles

tr = bodhan_transcribe(sample_wav, language="hi", client=client)
print(f"Transcript ({tr.duration_seconds}s, {tr.chunk_count} chunk(s), ₹{tr.cost_inr:.4f}):\n  {tr.text}")

subs = bodhan_make_subtitles(text=tr.text, duration_seconds=tr.duration_seconds, fmt="srt")
print(f"\nFormatted SRT ({subs['cue_count']} cue(s)):\n{subs['subtitles']}")

## 5. Translation (`bodhan_translate`) & Verification (`bodhan_verify_translation`)

Translates between English and 22 Indian languages (including canonical script variants such as `ks-Deva`, `mni-Beng`, `sd-Arab`) while preserving Markdown tables, refusing ASR-only `bhb`/`bho`, and verifying script ratio, bidirectional Indic/ASCII digits, and table integrity.

In [ ]:
from bodhan_mcp.tools import bodhan_translate, bodhan_verify_translation

table_md = (
    "# Time Table\n"
    "| Day | Subject | Room |\n"
    "|---|---|---|\n"
    "| Monday | Mathematics | 101 |\n"
    "| Tuesday | Science | 204 |"
)

trans = bodhan_translate(text=table_md, target_language="mr", client=client)
print(f"Marathi Translation ({trans.output_tokens} tokens, ₹{trans.cost_inr:.4f}):\n{trans.translation}")

v_res = bodhan_verify_translation(source_text=table_md, translated_text=trans.translation, target_language="mr", client=client)
print("\nVerification result:", v_res)

## 6. Speech Synthesis (`bodhan_speak`) & Back-Transcription Verification (`bodhan_verify_speech`)

`bodhan_speak` validates language & voice against the 45-voice catalog, splits long inputs into <= 220-char chunks, joins PCM16 24kHz mono WAV frames, and supports `verify=True` (with cache-bypassing repair retry).

In [ ]:
from bodhan_mcp.tools import bodhan_speak, bodhan_verify_speech

out_wav = str(RECIPE_DIR / "outputs" / "walkthrough_hindi_speech.wav")
paragraph = "आज का मुख्य समाचार। राज्य में नई शिक्षा नीति लागू हो गई है। सभी छात्र खुश हैं।"

sp = bodhan_speak(text=paragraph, language="hi", voice="Kavya", output_file=out_wav, client=client)
print(f"Synthesized: {sp.audio_path} ({sp.duration_seconds}s, {sp.character_count} chars, ₹{sp.cost_inr:.4f})")

vs = bodhan_verify_speech(audio_file=sp.audio_path, expected_text=paragraph, language="hi", client=client)
print("Speech verification:", vs)

## 7. Document OCR (`bodhan_ocr`)

Extracts Markdown, tables, and layout blocks from images, folders of images, or multi-page PDFs (rasterized via pinned `pypdfium2==4.30.0`), while rejecting non-image inputs and filtering page furniture blocks.

In [ ]:
from bodhan_mcp.tools import bodhan_ocr

worksheet_img = str(RECIPE_DIR / "sample_data" / "worksheet.png")
ocr_res = bodhan_ocr(worksheet_img, table_format="markdown", client=client)
print(f"Pages: {ocr_res.page_count} | Blocks: {len(ocr_res.blocks)} | Cost: ₹{ocr_res.cost_inr:.4f}")
print("\nExtracted Markdown preview:\n", ocr_res.markdown[:400])

## 8. Autonomous Task Agent, Security Guards & Upfront Budget Control

Run `TaskAgent` end-to-end on:
1. A multi-step task (Hindi voice note → English SRT subtitles)
2. An impossible task (French video → honest `impossible_task` at ₹0.00)
3. An upfront cost-budget cap (`--max-cost-inr 0.30` on a 4-page PDF → stops at ₹0.00 before overspending)
4. Code-level Ground Rule 1 provenance & path-traversal enforcement

In [ ]:
from bodhan_agent.agent import TaskAgent

# 1. Multi-step task
out_sub = RECIPE_DIR / "outputs" / "walkthrough_subtitles"
agent_sub = TaskAgent(
    task="Transcribe this Hindi voice note and give me English subtitles",
    files_path=sample_wav,
    out_dir=str(out_sub),
    max_steps=20,
    max_cost_inr=5.0,
)
res_sub = agent_sub.run()
print("1) Subtitles task:", res_sub["status"], "| artifacts:", [Path(a).name for a in res_sub["artifacts"]])

# 2. Impossible task (unsupported language -> ₹0.00)
out_imp = RECIPE_DIR / "outputs" / "walkthrough_impossible"
agent_imp = TaskAgent(
    task="Translate this video into French",
    files_path=str(RECIPE_DIR / "sample_data" / "sample_video.webm"),
    out_dir=str(out_imp),
)
res_imp = agent_imp.run()
print("2) Impossible task:", res_imp["status"], "|", res_imp["message"][:95], "...")

# 3. Upfront budget guard: 4-page PDF (~₹0.80) with ₹0.30 cap -> halts at ₹0.00
out_bgt = RECIPE_DIR / "outputs" / "walkthrough_budget"
agent_bgt = TaskAgent(
    task="Make a Tamil audio summary of this science chapter PDF",
    files_path=sample_pdf,
    out_dir=str(out_bgt),
    max_cost_inr=0.30,
)
res_bgt = agent_bgt.run()
rep_bgt = json.loads((out_bgt / "report.json").read_text(encoding="utf-8"))
print(f"3) Upfront budget guard: status={res_bgt['status']} | spent=₹{rep_bgt['total_cost_inr']:.4f} (limit=₹0.30)")

# 4. Code-level Ground Rule 1 & path traversal defense
try:
    agent_sub.execute_tool("bodhan_save_artifact", {"filename": "../../escape.txt", "content": "ok"}, "test")
except ValueError as err:
    print("4) Path traversal blocked:", err)